# Create Dummy files

In [ ]:
# !pip install reportlab openpyxl

In [ ]:
import os
import json
import pandas as pd
from docx import Document
from openpyxl import Workbook

# Create sample_docs directory if not exists
os.makedirs("sample_docs", exist_ok=True)

# 1. TXT file
with open("sample_docs/sample.txt", "w", encoding="utf-8") as f:
    f.write("This is a sample text file for testing.\nIt contains multiple lines of dummy content.\nRAG testing.")

# 2. Markdown file
with open("sample_docs/sample.md", "w", encoding="utf-8") as f:
    f.write("# Sample Markdown File\n\nThis is a **markdown** file.\n\n- Bullet 1\n- Bullet 2\n")

# 3. PDF file
from reportlab.pdfgen import canvas

pdf_path = "sample_docs/sample.pdf"
c = canvas.Canvas(pdf_path)
c.drawString(100, 750, "Sample PDF File")
c.drawString(100, 730, "This PDF contains dummy text for testing RAG applications.")
c.save()

# 4. CSV file
df_csv = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "score": [85, 90, 95]
})
df_csv.to_csv("sample_docs/sample.csv", index=False)

# 5. Excel file (xlsx)
wb = Workbook()
ws = wb.active
ws.title = "Sheet1"
ws.append(["ID", "Name", "Score"])
ws.append([1, "Alice", 85])
ws.append([2, "Bob", 90])
ws.append([3, "Charlie", 95])
wb.save("sample_docs/sample.xlsx")

# 6. DOCX file
doc = Document()
doc.add_heading("Sample DOCX File", 0)
doc.add_paragraph("This is a sample Word document with dummy content.")
doc.add_paragraph("It will be used for testing document processing.")
doc.save("sample_docs/sample.docx")

# 7. HTML file
html_content = """
<!DOCTYPE html>
<html>
<head><title>Sample HTML</title></head>
<body>
<h1>Sample HTML File</h1>
<p>This is a paragraph in an HTML file.</p>
<ul>
  <li>List item 1</li>
  <li>List item 2</li>
</ul>
</body>
</html>
"""
with open("sample_docs/sample.html", "w", encoding="utf-8") as f:
    f.write(html_content)

# 8. JSON file
json_data = {
    "title": "Sample JSON",
    "description": "This is a sample JSON file with dummy content.",
    "items": [
        {"id": 1, "name": "Alice"},
        {"id": 2, "name": "Bob"}
    ]
}
with open("sample_docs/sample.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, indent=4)

"✅ All sample files created successfully inside 'sample_docs' folder."


# Test File Processor

In [ ]:
from src.file_processor import DocumentProcessor

def test_all_files():
    processor = DocumentProcessor()
    file_paths = [
        'sample_docs/sample.txt',
        'sample_docs/sample.md',
        'sample_docs/sample.pdf',
        'sample_docs/sample.csv',
        'sample_docs/sample.xlsx',
        'sample_docs/sample.docx',
        'sample_docs/sample.html',
        'sample_docs/sample.json'
    ]
    for path in file_paths:
        try:
            result = processor.process_document(path)
            assert 'content' in result and 'metadata' in result
            print(f"SUCCESS: {path} processed with length {len(result['content'])}")
        except Exception as e:
            print(f"FAILED: {path} -> {e}")

if __name__ == "__main__":
    test_all_files()


# Read Files

In [ ]:
from src.file_processor import DocumentProcessor

def read_and_print_files(file_paths):
    processor = DocumentProcessor()
    
    for path in file_paths:
        try:
            result = processor.process_document(path)
            
            content = result['content']
            metadata = result['metadata']

            print("\n" + "="*80)
            print(f"📄 File: {metadata['filename']}")
            print(f"📍 Path: {metadata['file_path']}")
            print(f"📂 Type: {metadata['file_type']}")
            print(f"📏 Size: {metadata['file_size']} bytes")
            print(f"🔢 Content length: {metadata['content_length']} characters")
            print("-"*80)
            print("📝 Extracted Content (first 500 chars):\n")
            print(content[:500])  # show preview
            print("="*80 + "\n")

        except Exception as e:
            print(f"❌ FAILED to process {path} -> {e}")

if __name__ == "__main__":
    file_paths = [
        'sample_docs/sample.txt',
        'sample_docs/sample.md',
        'sample_docs/sample.pdf',
        'sample_docs/sample.csv',
        'sample_docs/sample.xlsx',
        'sample_docs/sample.docx',
        'sample_docs/sample.html',
        'sample_docs/sample.json'
    ]
    read_and_print_files(file_paths)

# Test Chunking 

In [ ]:
from src.chunker import Chunker
from src.overlap_optimizer import OverlapOptimizer


def test_semantic_chunking():
    """Test semantic chunking functionality"""
    print("🧪 Testing Semantic Chunking...")

    sample_text = """
    This is the first sentence. This is the second sentence with more content.
    This is the third sentence that adds even more detail to our test.

    This is a new paragraph. It should be handled properly by the chunker.
    The chunker should maintain context across chunk boundaries.
    """

    metadata = {
        'filename': 'test.txt',
        'file_type': '.txt',
        'file_size': len(sample_text)
    }

    chunker = Chunker(chunk_size=50, overlap=15, strategy="semantic")
    chunks = chunker.semantic_chunking(sample_text, metadata)  # ✅ use public method

    print(f"Created {len(chunks)} chunks")
    for i, chunk in enumerate(chunks):
        chunk['metadata']['chunk_index'] = i  # ✅ ensure index exists
        chunk['metadata']['chunk_word_count'] = len(chunk['content'].split())

        print(f"\nChunk {i}: {chunk['metadata']['chunk_word_count']} words")
        print(f"Content preview: {chunk['content'][:80]}...")
        print(f"Metadata: {chunk['metadata']}")


def test_fixed_chunking():
    """Test fixed-size chunking functionality"""
    print("\n🧪 Testing Fixed-Size Chunking...")

    sample_text = "word " * 200  # 200 words

    metadata = {
        'filename': 'test_fixed.txt',
        'file_type': '.txt',
        'file_size': len(sample_text)
    }

    chunker = Chunker(chunk_size=50, overlap=10, strategy="fixed")
    chunks = chunker.fixed_size_shunking(sample_text, metadata)  # ✅ call fixed method

    print(f"Created {len(chunks)} chunks")
    for i, chunk in enumerate(chunks):
        chunk['metadata']['chunk_index'] = i  # ✅ ensure index exists
        chunk['metadata']['chunk_word_count'] = len(chunk['content'].split())

        print(f"\nChunk {i}: {chunk['metadata']['chunk_word_count']} words")
        print(f"Start-End: {chunk['metadata'].get('start_word_index', 'N/A')}-"
              f"{chunk['metadata'].get('end_word_index', 'N/A')}")
        print(f"Metadata: {chunk['metadata']}")


def test_overlap_optimizer():
    """Test overlap optimization logic"""
    print("\n🧪 Testing Overlap Optimizer...")

    optimizer = OverlapOptimizer()

    test_cases = [
        (1000, 500, '.txt'),
        (5000, 1000, '.pdf'),
        (500, 200, '.csv'),
        (10000, 1000, '.docx')
    ]

    for content_length, chunk_size, file_type in test_cases:
        overlap = optimizer.decide_overlap_size(content_length, chunk_size, file_type)
        print(f"Content: {content_length}, Chunk: {chunk_size}, Type: {file_type} → Overlap: {overlap}")


if __name__ == "__main__":
    test_semantic_chunking()
    test_fixed_chunking()
    test_overlap_optimizer()


In [ ]:
1000 * 0.25

# test_phase2.py

In [ ]:
from src.pipeline_manager import DocumentProcessingPipeline
from src.logger import get_logger

# Test the corrected pipeline
pipeline = DocumentProcessingPipeline(
    strategy="semantic", 
    chunk_size=200, 
    overlap=50
)

# Create a test file
import os
os.makedirs('sample_docs', exist_ok=True)
with open('sample_docs/test.txt', 'w') as f:
    f.write("This is a test document. " * 50)

# Test processing
chunks = pipeline.process_document_to_chunks('sample_docs/sample.txt')
print(f"✅ Successfully created {len(chunks)} chunks!")
for i, chunk in enumerate(chunks[:2]):  # Show first 2 chunks
    print(f"Chunk {i}: {chunk['metadata']['chunk_token_count']} tokens")


In [ ]:
from src.pipeline_manager import DocumentProcessingPipeline

# Initialize pipeline
pipeline = DocumentProcessingPipeline(
    strategy="semantic", 
    chunk_size=500, 
    overlap=100
)

# Process your real PDF
chunks = pipeline.process_document_to_chunks('sample_docs/1.pdf')
print(f"✅ Successfully created {len(chunks)} chunks!")

for i, chunk in enumerate(chunks):  # Show first 5 chunks
    print(f"\nChunk {i}: {chunk['metadata']['chunk_token_count']} tokens")
    print(f"Preview: {chunk['content']}...")  # Fixed: added [:100] to limit preview
    
    # View all metadata keys
    print(f"Available metadata keys: {list(chunk['metadata'].keys())}")
    
    # Check for page information (won't exist yet with current implementation)
    if 'start_page_number' in chunk['metadata']:
        print(f"Page: {chunk['metadata']['start_page_number']}")
    else:
        print("❌ No page information available (need to implement page-aware processing)")
    
    print("---")


In [ ]:
from src.pipeline_manager import DocumentProcessingPipeline

# Initialize pipeline
pipeline = DocumentProcessingPipeline(
    strategy="semantic", 
    chunk_size=500, 
    overlap=100
)

# Process your PDF
chunks = pipeline.process_document_to_chunks('sample_docs/1.pdf')
print(f"🎉 Successfully created {len(chunks)} chunks from PDF!")

# Document overview
if chunks:
    print(f"📖 Document: {chunks[0]['metadata']['filename']}")
    print(f"📄 Total pages: {chunks[0]['metadata']['total_pages']}")
    print(f"📏 Document size: {chunks[0]['metadata']['file_size']} bytes")

print("\n" + "="*60)

# Detailed chunk analysis
for i, chunk in enumerate(chunks[:5]):  # Show first 5 chunks
    metadata = chunk['metadata']
    
    print(f"\n📄 Chunk {i+1}:")
    print(f"   🔤 Tokens: {metadata['chunk_token_count']}")
    print(f"   📍 Page: {metadata['start_page_number']}", end="")
    
    if metadata.get('spans_multiple_pages', False):
        print(f" → {metadata['end_page_number']} (spans multiple pages)")
    else:
        print(f" (single page)")
    
    print(f"   📊 Position: chars {metadata['char_start_in_document']}-{metadata['char_end_in_document']}")
    print(f"   🎯 Strategy: {metadata['chunking_strategy']}")
    print(f"   📝 Preview: {chunk['content'][:120]}...")
    print("   " + "-"*50)

# Page distribution analysis
print(f"\n📊 Page Distribution Analysis:")
page_chunks = {}
for chunk in chunks:
    page = chunk['metadata']['start_page_number']
    page_chunks[page] = page_chunks.get(page, 0) + 1

for page in sorted(page_chunks.keys()):
    print(f"   Page {page}: {page_chunks[page]} chunks")

# Cross-page chunks
cross_page_chunks = [c for c in chunks if c['metadata'].get('spans_multiple_pages', False)]
if cross_page_chunks:
    print(f"\n🔗 {len(cross_page_chunks)} chunks span multiple pages:")
    for chunk in cross_page_chunks[:3]:  # Show first 3
        meta = chunk['metadata']
        print(f"   Chunk {meta['chunk_index']}: Pages {meta['start_page_number']}-{meta['end_page_number']}")


# test_simple_embedding.py (Easy Testing)


In [ ]:
import os
from src.embedding_pipeline import EmbeddingPipeline

def test_simple_embedding():
    """Test the simplified embedding pipeline"""
    print("🧪 Testing Simplified Embedding Pipeline with all-mpnet-base-v2")
    
    # Create or use existing test document
    if not os.path.exists('sample_docs/1.pdf'):
        # Create a text file for testing
        os.makedirs('sample_docs', exist_ok=True)
        test_content = """
        Machine learning is a method of data analysis that automates analytical model building.
        It is a branch of artificial intelligence based on the idea that systems can learn from data,
        identify patterns and make decisions with minimal human intervention.
        
        Deep learning is a subset of machine learning that uses neural networks with multiple layers.
        These neural networks attempt to simulate the behavior of the human brain to learn from large amounts of data.
        
        Natural language processing helps computers understand, interpret and manipulate human language.
        It draws from many disciplines, including computer science and computational linguistics.
        """ * 3
        
        with open('sample_docs/ai_test.txt', 'w') as f:
            f.write(test_content)
        file_path = 'sample_docs/ai_test.txt'
    else:
        file_path = 'sample_docs/1.pdf'
    
    # Initialize simplified pipeline
    pipeline = EmbeddingPipeline(
        chunking_strategy="semantic",
        chunk_size=300,  # Good size for all-mpnet-base-v2
        overlap=50,
        batch_size=16
    )
    
    # Process document
    print(f"📄 Processing: {file_path}")
    results = pipeline.process_single_document(file_path)
    
    print(f"\n✅ Results:")
    print(f"   Chunks created: {results['chunk_count']}")
    print(f"   Embeddings shape: {results['embeddings'].shape}")
    print(f"   Embedding dimension: {results['embedding_dimension']}")
    
    # Test semantic search
    test_queries = [
        "What is machine learning?",
        "How does deep learning work?",
        "What is natural language processing?"
    ]
    
    for query in test_queries:
        print(f"\n🔍 Query: '{query}'")
        similar_chunks = pipeline.search_similar_chunks(results, query, top_k=3)
        
        for i, (similarity, chunk) in enumerate(similar_chunks[:2]):
            print(f"   {i+1}. Similarity: {similarity:.3f}")
            print(f"      Page: {chunk['metadata'].get('start_page_number', 'N/A')}")
            print(f"      Preview: {chunk['content'][:100]}...")
            print()

if __name__ == "__main__":
    test_simple_embedding()


# 1. Test with Better Content


In [ ]:
# Add this test with content-rich text
def test_with_better_content():
    test_content = """
    Machine learning is a powerful approach to data analysis that enables computers to learn and make predictions from data without being explicitly programmed for every scenario. It works by using algorithms that can identify patterns in data and use these patterns to make predictions about new, unseen data.

    There are three main types of machine learning: supervised learning, where algorithms learn from labeled training data; unsupervised learning, where patterns are found in data without labels; and reinforcement learning, where systems learn through interaction and feedback.

    Deep learning is a subset of machine learning that uses artificial neural networks with multiple layers to model and understand complex patterns in data. These networks are inspired by the human brain and can automatically learn hierarchical representations of data.

    Natural language processing is a field that combines computational linguistics with machine learning to help computers understand, interpret, and generate human language. It involves tasks like sentiment analysis, language translation, and text summarization.
    """
    
    os.makedirs('sample_docs', exist_ok=True)
    with open('sample_docs/ml_content.txt', 'w') as f:
        f.write(test_content)
    
    # Test with this content-rich file
    pipeline = EmbeddingPipeline(chunk_size=200, overlap=30)
    results = pipeline.process_single_document('sample_docs/ml_content.txt')
    
    # Test queries
    for query in ["What is machine learning?", "How does deep learning work?"]:
        similar_chunks = pipeline.search_similar_chunks(results, query, top_k=2)
        print(f"\nQuery: {query}")
        for sim, chunk in similar_chunks:
            print(f"  Similarity: {sim:.3f}")
            print(f"  Content: {chunk['content'][:100]}...")

if __name__ == "__main__":
    test_with_better_content()

# test_vector_store.py

In [ ]:
import os
import numpy as np
from src.rag_system import RAGSystem

def test_rag_system():
    """Test the complete RAG system"""
    print("🧪 Testing Complete RAG System with Vector Database")
    
    # Initialize RAG system
    rag = RAGSystem(
        collection_name="test_rag",
        chunk_size=150,  # SMALLER chunks for better matching
        overlap=30       # Less overlap
    )
    
    # Create test documents
    os.makedirs('sample_docs', exist_ok=True)
    
    # Document 1: ML Content
    ml_content = """
    Machine learning is a powerful subset of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed. The core idea is to develop algorithms that can automatically identify patterns in data and use these patterns to make predictions or decisions about new, unseen data.

    There are three main types of machine learning approaches. Supervised learning uses labeled training data to learn a mapping from inputs to outputs. Unsupervised learning finds hidden patterns in data without labels. Reinforcement learning learns through trial and error by interacting with an environment and receiving feedback in the form of rewards or penalties.

    Deep learning represents a revolutionary advance in machine learning, using artificial neural networks with multiple layers to model complex patterns in data. These deep networks can automatically learn hierarchical representations, making them particularly effective for tasks like image recognition, natural language processing, and speech recognition.
    """
    
    with open('sample_docs/ml_guide.txt', 'w') as f:
        f.write(ml_content)
    
    # Document 2: AI Content  
    ai_content = """
    Artificial Intelligence is the simulation of human intelligence in machines that are programmed to think and learn like humans. AI systems can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.

    Natural Language Processing is a branch of AI that helps computers understand, interpret, and manipulate human language. NLP combines computational linguistics with statistical machine learning to enable computers to process and analyze large amounts of natural language data.

    Computer vision is another important AI field that enables machines to interpret and understand visual information from the world. This technology is used in applications ranging from medical image analysis to autonomous vehicles and facial recognition systems.
    """
    
    with open('sample_docs/ai_overview.txt', 'w') as f:
        f.write(ai_content)
    
    print("📄 Created test documents")
    
    # Add documents to RAG system
    file_paths = ['sample_docs/ml_guide.txt', 'sample_docs/ai_overview.txt']
    document_ids = rag.add_documents(file_paths)
    
    print(f"✅ Added documents to RAG system:")
    for file_path, ids in document_ids.items():
        print(f"   {file_path}: {len(ids)} chunks")
    
    # Test queries
    test_queries = [
        "What is machine learning?",
        "How does deep learning work?", 
        "What is natural language processing?",
        "Explain supervised learning",
        "What are the types of machine learning?"
    ]
    
    print(f"\n🔍 Testing queries:")
    for query in test_queries:
        print(f"\n📋 Query: '{query}'")
        
        results = rag.query(query, top_k=3)
        
        print(f"   📊 Retrieved {results['chunk_count']} chunks")
        print(f"   🎯 Average similarity: {results['avg_similarity']}")
        
        # Show top citations
        for citation in results['citations'][:2]:
            print(f"   📖 [{citation['id']}] {citation['filename']} (similarity: {citation['similarity_score']})")
            print(f"      Preview: {citation['content_preview'][:80]}...")
    
    # Test system stats
    print(f"\n📊 System Statistics:")
    stats = rag.get_system_stats()
    print(f"   Total documents: {stats['vector_store']['total_documents']}")
    print(f"   Source files: {stats['vector_store']['unique_source_files']}")
    print(f"   Embedding model: {stats['embedding_model']['model_name']}")
    print(f"   Chunk size: {stats['chunking_config']['chunk_size']} tokens")
    
    # Test source-specific query
    print(f"\n🎯 Testing source-specific query:")
    ml_results = rag.query(
        "What is supervised learning?", 
        source_filter='sample_docs/ml_guide.txt',
        top_k=2
    )
    print(f"   ML guide results: {ml_results['chunk_count']} chunks")
    print(f"   Avg similarity: {ml_results['avg_similarity']}")

if __name__ == "__main__":
    test_rag_system()


# AB_testing reranking.py


In [1]:
import os
from src.rag_system import RAGSystem
from src.enhanced_retrieval_engine import EnhancedRetrievalEngine
from src.vector_store import VectorStore
from src.embedding_manager import EmbeddingManager
import time
from typing import List, Dict, Any, Annotated

class RerankerABTest:
    """
    A/B testing framework to compare retrieval with and without reranking
    """
    
    def __init__(self):
        # Initialize shared components
        self.vector_store = VectorStore(collection_name="rerank_test")
        self.embedding_manager = EmbeddingManager()
        
        # Create two retrieval engines
        self.retriever_with_rerank = EnhancedRetrievalEngine(
            vector_store=self.vector_store,
            embedding_manager=self.embedding_manager,
            enable_reranking=True
        )
        
        self.retriever_without_rerank = EnhancedRetrievalEngine(
            vector_store=self.vector_store,
            embedding_manager=self.embedding_manager,
            enable_reranking=False
        )
        
        print("🧪 A/B Test Framework Initialized")
        print("   Group A: Standard Retrieval")
        print("   Group B: Retrieval + Reranking")

    def run_comparison_test(self, test_queries: List[str]) -> Dict:
        """Run A/B comparison test"""
        
        results = {
            'without_reranking': [],
            'with_reranking': [],
            'performance_summary': {}
        }
        
        print(f"\n🚀 Running A/B test with {len(test_queries)} queries")
        
        for i, query in enumerate(test_queries, 1):
            print(f"\n📋 Query {i}/{len(test_queries)}: '{query}'")
            
            # Test without reranking
            start_time = time.time()
            results_no_rerank = self.retriever_without_rerank.retrieve_context(
                query, top_k=5, fetch_k=5
            )
            time_no_rerank = time.time() - start_time
            
            # Test with reranking  
            start_time = time.time()
            results_rerank = self.retriever_with_rerank.retrieve_context(
                query, top_k=5, fetch_k=20  # Fetch more for reranking
            )
            time_rerank = time.time() - start_time
            
            # Store results
            results['without_reranking'].append({
                'query': query,
                'chunk_count': results_no_rerank['chunk_count'],
                'avg_similarity': results_no_rerank['avg_similarity'],
                'processing_time': time_no_rerank,
                'citations': results_no_rerank['citations']
            })
            
            results['with_reranking'].append({
                'query': query,
                'chunk_count': results_rerank['chunk_count'],
                'avg_similarity': results_rerank['avg_similarity'],
                'processing_time': time_rerank,
                'citations': results_rerank['citations']
            })
            
            # Print comparison
            print(f"   📊 Without Reranking: {results_no_rerank['chunk_count']} chunks, {results_no_rerank['avg_similarity']:.3f} avg sim, {time_no_rerank:.3f}s")
            print(f"   🎯 With Reranking:    {results_rerank['chunk_count']} chunks, {results_rerank['avg_similarity']:.3f} avg sim, {time_rerank:.3f}s")
            
            if results_rerank['avg_similarity'] > results_no_rerank['avg_similarity']:
                improvement = ((results_rerank['avg_similarity'] - results_no_rerank['avg_similarity']) / results_no_rerank['avg_similarity']) * 100
                print(f"   ✅ Reranking improved similarity by {improvement:.1f}%")
            else:
                print(f"   ❌ Reranking did not improve similarity")
        
        # Calculate summary metrics
        results['performance_summary'] = self._calculate_summary(results)
        
        return results
    
    def _calculate_summary(self, results: Dict) -> Dict:
        """Calculate performance summary metrics"""
        
        no_rerank = results['without_reranking']
        with_rerank = results['with_reranking']
        
        summary = {
            'avg_similarity_no_rerank': sum(r['avg_similarity'] for r in no_rerank) / len(no_rerank),
            'avg_similarity_with_rerank': sum(r['avg_similarity'] for r in with_rerank) / len(with_rerank),
            'avg_time_no_rerank': sum(r['processing_time'] for r in no_rerank) / len(no_rerank),
            'avg_time_with_rerank': sum(r['processing_time'] for r in with_rerank) / len(with_rerank),
            'queries_improved_by_rerank': sum(1 for i in range(len(no_rerank)) 
                                            if with_rerank[i]['avg_similarity'] > no_rerank[i]['avg_similarity']),
            'total_queries': len(no_rerank)
        }
        
        summary['similarity_improvement_pct'] = ((summary['avg_similarity_with_rerank'] - summary['avg_similarity_no_rerank']) / summary['avg_similarity_no_rerank']) * 100
        summary['time_overhead_pct'] = ((summary['avg_time_with_rerank'] - summary['avg_time_no_rerank']) / summary['avg_time_no_rerank']) * 100
        summary['queries_improved_pct'] = (summary['queries_improved_by_rerank'] / summary['total_queries']) * 100
        
        return summary

def test_reranking_comparison():
    """Test reranking vs no reranking"""
    
    # Setup test data
    os.makedirs('sample_docs', exist_ok=True)
    
    # Create detailed test content
    detailed_ml_content = """
    Supervised learning is a machine learning approach where algorithms learn from labeled training data. In supervised learning, you provide the model with input-output pairs, and it learns to map inputs to correct outputs. Common supervised learning algorithms include linear regression, decision trees, random forests, and support vector machines. Examples include email spam detection, image classification, and price prediction.

    Unsupervised learning finds patterns in data without labeled examples. It includes clustering algorithms like K-means that group similar data points, and dimensionality reduction techniques like Principal Component Analysis (PCA) that simplify data while preserving important information. Common applications include customer segmentation and data compression.

    Reinforcement learning learns through trial and error, receiving rewards or penalties for actions. The agent explores an environment and learns optimal strategies to maximize cumulative reward. This approach is used in game playing (like AlphaGo), robotics, and autonomous vehicle navigation.

    Deep learning uses artificial neural networks with multiple layers to model complex patterns in data. These networks can automatically learn hierarchical representations, making them effective for image recognition, natural language processing, and speech recognition. Popular architectures include convolutional neural networks for images and recurrent neural networks for sequences.
    """
    
    ai_detailed_content = """
    Natural Language Processing combines computational linguistics with machine learning to help computers understand human language. Key NLP tasks include sentiment analysis, named entity recognition, machine translation, and text summarization. Modern NLP systems use transformer architectures like BERT and GPT for improved understanding.

    Computer vision enables machines to interpret visual information from images and videos. Core techniques include object detection, image segmentation, and facial recognition. Convolutional neural networks are the foundation of most computer vision systems, with applications in medical imaging, autonomous vehicles, and security systems.

    Artificial Intelligence encompasses the broader goal of creating intelligent machines. AI includes symbolic reasoning, expert systems, and modern machine learning approaches. Current AI systems excel at specific tasks but artificial general intelligence remains a long-term research goal.
    """
    
    with open('sample_docs/detailed_ml.txt', 'w') as f:
        f.write(detailed_ml_content)
    
    with open('sample_docs/detailed_ai.txt', 'w') as f:
        f.write(ai_detailed_content)
    
    # Initialize RAG system and add documents
    rag = RAGSystem(collection_name="rerank_test", chunk_size=200, overlap=40)
    file_paths = ['sample_docs/detailed_ml.txt', 'sample_docs/detailed_ai.txt']
    rag.add_documents(file_paths)
    
    # Initialize A/B test
    ab_test = RerankerABTest()
    
    # Define test queries
    test_queries = [
        "What is supervised learning and how does it work?",
        "Explain the difference between supervised and unsupervised learning",
        "How does reinforcement learning work with rewards?", 
        "What are the applications of computer vision?",
        "What is natural language processing used for?",
        "How do neural networks work in deep learning?",
        "What algorithms are used in supervised learning?",
        "What is the goal of artificial intelligence?"
    ]
    
    # Run A/B comparison
    results = ab_test.run_comparison_test(test_queries)
    
    # Display detailed results
    print("\n" + "="*80)
    print("🏁 A/B TEST RESULTS SUMMARY")
    print("="*80)
    
    summary = results['performance_summary']
    
    print(f"📊 Overall Metrics:")
    print(f"   Average Similarity (No Rerank):  {summary['avg_similarity_no_rerank']:.3f}")
    print(f"   Average Similarity (Reranked):   {summary['avg_similarity_with_rerank']:.3f}")
    print(f"   Similarity Improvement:          {summary['similarity_improvement_pct']:+.1f}%")
    print(f"")
    print(f"⏱️  Performance Metrics:")
    print(f"   Average Time (No Rerank):        {summary['avg_time_no_rerank']:.3f}s")
    print(f"   Average Time (Reranked):         {summary['avg_time_with_rerank']:.3f}s")
    print(f"   Time Overhead:                   {summary['time_overhead_pct']:+.1f}%")
    print(f"")
    print(f"🎯 Quality Metrics:")
    print(f"   Queries Improved by Reranking:   {summary['queries_improved_by_rerank']}/{summary['total_queries']}")
    print(f"   Improvement Rate:                {summary['queries_improved_pct']:.1f}%")
    
    if summary['similarity_improvement_pct'] > 5:
        print(f"\n✅ RECOMMENDATION: Enable reranking - significant improvement detected!")
    elif summary['similarity_improvement_pct'] > 0:
        print(f"\n⚡ RECOMMENDATION: Consider reranking - modest improvement with time overhead")
    else:
        print(f"\n❌ RECOMMENDATION: Skip reranking - no significant benefit for this dataset")
    
    print("="*80)


if __name__ == "__main__":
    test_reranking_comparison()


d:\AIML And DSA\AI Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO 2025-09-14 21:44:14,408 src.pipeline_manager] Document processing pipeline initialized with token-aware chunker
[INFO 2025-09-14 21:44:14,409 src.embedding_manager] Loding all-mpnet-base-v2-model...
[INFO 2025-09-14 21:44:19,701 src.embedding_manager] Model Loaded Successfully, embedding dimention: 768
[INFO 2025-09-14 21:44:19,703 src.embedding_manager] EmbeddingManager initialized with all-mpnet-base-v2
[INFO 2025-09-14 21:44:19,705 src.embedding_pipeline] EmbeddingPipeline initialized:
[INFO 2025-09-14 21:44:19,706 src.embedding_pipeline]   Chunking: semantic, size=200, overlap=40
[INFO 2025-09-14 21:44:19,706 src.embedding_pipeline]   Embedding: all-mpnet-base-v2
[INFO 2025-09-14 21:44:19,892 src.vector_store] Found exi

🧪 A/B Test Framework Initialized
   Group A: Standard Retrieval
   Group B: Retrieval + Reranking

🚀 Running A/B test with 8 queries

📋 Query 1/8: 'What is supervised learning and how does it work?'


[INFO 2025-09-14 21:44:35,775 src.vector_store] Found 9 similar documents
[INFO 2025-09-14 21:44:35,775 src.reranker] Reranking 3 documents for query: 'What is supervised learning and how does it work?...'
[INFO 2025-09-14 21:44:35,872 src.reranker] Reranking complete: 3 docs, top score = 7.772
[INFO 2025-09-14 21:44:35,873 src.enhanced_retrieval_engine] 🔍 Retrieving context: query='Explain the difference between supervised and unsu...', fetch_k=5, final_k=5
[INFO 2025-09-14 21:44:35,873 src.embedding_manager] Generating  embeddings for len1 texts
[INFO 2025-09-14 21:44:35,920 src.embedding_manager] Generated 1 embeddings in 0.05s (-21.6) texts/sec
[INFO 2025-09-14 21:44:35,922 src.vector_store] Found 5 similar documents
[INFO 2025-09-14 21:44:35,923 src.enhanced_retrieval_engine] 🔍 Retrieving context: query='Explain the difference between supervised and unsu...', fetch_k=20, final_k=5
[INFO 2025-09-14 21:44:35,923 src.embedding_manager] Generating  embeddings for len1 texts
[INFO 2025

   📊 Without Reranking: 3 chunks, 0.469 avg sim, 0.139s
   🎯 With Reranking:    3 chunks, 7.772 avg sim, 0.167s
   ✅ Reranking improved similarity by 1557.1%

📋 Query 2/8: 'Explain the difference between supervised and unsupervised learning'


[INFO 2025-09-14 21:44:36,080 src.enhanced_retrieval_engine] 🔍 Retrieving context: query='How does reinforcement learning work with rewards?...', fetch_k=5, final_k=5
[INFO 2025-09-14 21:44:36,080 src.embedding_manager] Generating  embeddings for len1 texts
[INFO 2025-09-14 21:44:36,120 src.embedding_manager] Generated 1 embeddings in 0.04s (-25.4) texts/sec
[INFO 2025-09-14 21:44:36,122 src.vector_store] Found 5 similar documents
[WARNING 2025-09-14 21:44:36,123 src.enhanced_retrieval_engine] No results above similarity threshold 0.1
[INFO 2025-09-14 21:44:36,123 src.enhanced_retrieval_engine] 🔍 Retrieving context: query='How does reinforcement learning work with rewards?...', fetch_k=20, final_k=5
[INFO 2025-09-14 21:44:36,123 src.embedding_manager] Generating  embeddings for len1 texts
[INFO 2025-09-14 21:44:36,162 src.embedding_manager] Generated 1 embeddings in 0.04s (-26.3) texts/sec
[INFO 2025-09-14 21:44:36,164 src.vector_store] Found 9 similar documents
[WARNING 2025-09-14 21:

   📊 Without Reranking: 3 chunks, 0.218 avg sim, 0.050s
   🎯 With Reranking:    3 chunks, 6.409 avg sim, 0.157s
   ✅ Reranking improved similarity by 2839.9%

📋 Query 3/8: 'How does reinforcement learning work with rewards?'
   📊 Without Reranking: 0 chunks, 0.000 avg sim, 0.043s
   🎯 With Reranking:    0 chunks, 0.000 avg sim, 0.043s
   ❌ Reranking did not improve similarity

📋 Query 4/8: 'What are the applications of computer vision?'
   📊 Without Reranking: 0 chunks, 0.000 avg sim, 0.039s
   🎯 With Reranking:    0 chunks, 0.000 avg sim, 0.040s
   ❌ Reranking did not improve similarity

📋 Query 5/8: 'What is natural language processing used for?'


[INFO 2025-09-14 21:44:36,284 src.vector_store] Found 5 similar documents
[INFO 2025-09-14 21:44:36,284 src.enhanced_retrieval_engine] 🔍 Retrieving context: query='What is natural language processing used for?...', fetch_k=20, final_k=5
[INFO 2025-09-14 21:44:36,285 src.embedding_manager] Generating  embeddings for len1 texts
[INFO 2025-09-14 21:44:36,327 src.embedding_manager] Generated 1 embeddings in 0.04s (-24.2) texts/sec
[INFO 2025-09-14 21:44:36,329 src.vector_store] Found 9 similar documents
[INFO 2025-09-14 21:44:36,330 src.reranker] Reranking 3 documents for query: 'What is natural language processing used for?...'
[INFO 2025-09-14 21:44:36,385 src.reranker] Reranking complete: 3 docs, top score = 4.715
[INFO 2025-09-14 21:44:36,386 src.enhanced_retrieval_engine] 🔍 Retrieving context: query='How do neural networks work in deep learning?...', fetch_k=5, final_k=5
[INFO 2025-09-14 21:44:36,387 src.embedding_manager] Generating  embeddings for len1 texts
[INFO 2025-09-14 21:44:3

   📊 Without Reranking: 3 chunks, 0.240 avg sim, 0.039s
   🎯 With Reranking:    3 chunks, 4.715 avg sim, 0.102s
   ✅ Reranking improved similarity by 1864.6%

📋 Query 6/8: 'How do neural networks work in deep learning?'
   📊 Without Reranking: 3 chunks, 0.165 avg sim, 0.039s
   🎯 With Reranking:    3 chunks, 7.956 avg sim, 0.077s
   ✅ Reranking improved similarity by 4721.8%

📋 Query 7/8: 'What algorithms are used in supervised learning?'


[INFO 2025-09-14 21:44:36,654 src.reranker] Reranking complete: 3 docs, top score = 8.932
[INFO 2025-09-14 21:44:36,655 src.enhanced_retrieval_engine] 🔍 Retrieving context: query='What is the goal of artificial intelligence?...', fetch_k=5, final_k=5
[INFO 2025-09-14 21:44:36,655 src.embedding_manager] Generating  embeddings for len1 texts
[INFO 2025-09-14 21:44:36,691 src.embedding_manager] Generated 1 embeddings in 0.04s (-28.0) texts/sec
[INFO 2025-09-14 21:44:36,694 src.vector_store] Found 5 similar documents
[WARNING 2025-09-14 21:44:36,694 src.enhanced_retrieval_engine] No results above similarity threshold 0.1
[INFO 2025-09-14 21:44:36,695 src.enhanced_retrieval_engine] 🔍 Retrieving context: query='What is the goal of artificial intelligence?...', fetch_k=20, final_k=5
[INFO 2025-09-14 21:44:36,695 src.embedding_manager] Generating  embeddings for len1 texts
[INFO 2025-09-14 21:44:36,730 src.embedding_manager] Generated 1 embeddings in 0.04s (-28.6) texts/sec
[INFO 2025-09-14 21

   📊 Without Reranking: 3 chunks, 0.367 avg sim, 0.039s
   🎯 With Reranking:    3 chunks, 8.932 avg sim, 0.113s
   ✅ Reranking improved similarity by 2333.8%

📋 Query 8/8: 'What is the goal of artificial intelligence?'
   📊 Without Reranking: 0 chunks, 0.000 avg sim, 0.040s
   🎯 With Reranking:    0 chunks, 0.000 avg sim, 0.040s
   ❌ Reranking did not improve similarity

🏁 A/B TEST RESULTS SUMMARY
📊 Overall Metrics:
   Average Similarity (No Rerank):  0.182
   Average Similarity (Reranked):   4.473
   Similarity Improvement:          +2352.6%

⏱️  Performance Metrics:
   Average Time (No Rerank):        0.054s
   Average Time (Reranked):         0.092s
   Time Overhead:                   +71.7%

🎯 Quality Metrics:
   Queries Improved by Reranking:   5/8
   Improvement Rate:                62.5%

✅ RECOMMENDATION: Enable reranking - significant improvement detected!


# Test Complete System

In [1]:
import os
import asyncio
from src.rag_generator import RAGGenerator
from src.logger import get_logger

logger = get_logger(__name__)

async def test_complete_rag_system():
    """Test the complete RAG system with Groq LLM"""
    
    print("🧪 Testing Complete RAG System with Groq LLM Integration")
    
    # Check environment
    if not os.getenv("GROQ_API_KEY"):
        print("❌ Please set GROQ_API_KEY environment variable")
        return
    
    try:
        # Initialize complete RAG system
        rag_generator = RAGGenerator(
            groq_model="openai/gpt-oss-120b",
            retrieval_top_k=3,
            enable_reranking=True  # Based on your proven A/B results
        )
        
        print("✅ RAG Generator initialized successfully")
        
        # Test with existing documents (from your previous tests)
        test_questions = [
            "What is machine learning and how does it work?",
            "Explain the difference between supervised and unsupervised learning",
            "What are the applications of deep learning in AI?",
            "How does natural language processing help computers understand human language?"
        ]
        
        print(f"\n🔍 Testing {len(test_questions)} questions:")
        
        for i, question in enumerate(test_questions, 1):
            print(f"\n📋 Question {i}: {question}")
            
            # Generate complete answer
            result = rag_generator.generate_answer(question)
            
            # Display results
            print(f"📊 Metadata:")
            print(f"   Chunks retrieved: {result['metadata']['chunks_retrieved']}")
            print(f"   Avg similarity: {result['metadata']['avg_similarity']:.3f}")
            print(f"   Generation time: {result['metadata']['generation_time']:.2f}s")
            print(f"   Total tokens: {result['metadata'].get('total_tokens', 'N/A')}")
            
            print(f"🎯 Answer: {result['answer'][:200]}...")
            
            print(f"📖 Sources:")
            for j, source in enumerate(result['sources'][:2], 1):
                print(f"   [{j}] {source['filename']} - Similarity: {source['similarity_score']:.3f}")
        
        # Test system stats
        print(f"\n📊 System Statistics:")
        stats = rag_generator.get_system_stats()
        print(f"   Total documents: {stats['rag_system']['vector_store']['total_documents']}")
        print(f"   LLM model: {stats['llm']['model']}")
        print(f"   Reranking enabled: {stats['configuration']['reranking_enabled']}")
        
        print(f"\n🎉 Complete RAG system test successful!")
        
    except Exception as e:
        logger.error(f"❌ Complete system test failed: {e}")
        print(f"❌ Test failed: {e}")

if __name__ == "__main__":
    asyncio.run(test_complete_rag_system())


d:\AIML And DSA\AI Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: asyncio.run() cannot be called from a running event loop